# 03 Baseline Random Forest

Train a classical baseline for Remaining Useful Life prediction.

In [1]:
from pathlib import Path
import json
import pickle
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error

ROOT = Path('..').resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.train import nasa_score

PROCESSED_DIR = ROOT / 'data' / 'processed'
MODELS_DIR = ROOT / 'models'
REPORTS_DIR = ROOT / 'reports' / 'figures'
MODELS_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_PATH = PROCESSED_DIR / 'train_fd001_features.parquet'
TEST_PATH = PROCESSED_DIR / 'test_fd001_features.parquet'

## 1) Load Engineered Features
Use feature tables created in notebook 02 (including temporal and scaled features).

In [2]:
train_df = pd.read_parquet(TRAIN_PATH)
test_df = pd.read_parquet(TEST_PATH)

feature_cols = [c for c in train_df.columns if c not in {'engine_id', 'cycle', 'rul'}]
X_train = train_df[feature_cols]
y_train = train_df['rul']
X_test = test_df[feature_cols]
y_test = test_df['rul']

print(f'train shape: {train_df.shape}')
print(f'test shape : {test_df.shape}')
print(f'num features: {len(feature_cols)}')

train shape: (20631, 97)
test shape : (13096, 97)
num features: 94


## 2) Train and Evaluate Random Forest
Fit baseline model and report RMSE, MAE, and NASA score.

In [ ]:
rf = RandomForestRegressor(
    n_estimators=400,
    max_depth=18,
    min_samples_leaf=2,
    n_jobs=-1,
    random_state=42,
    )
rf.fit(X_train, y_train)

pred = rf.predict(X_test)
rmse = float(np.sqrt(mean_squared_error(y_test, pred)))
mae = float(mean_absolute_error(y_test, pred))
score = float(nasa_score(y_test.to_numpy(), pred))

metrics = {'rmse': rmse, 'mae': mae, 'nasa_score': score}
print(metrics)

## 3) Diagnostics and Artifact Export
Plot prediction quality and save baseline artifacts for reproducibility.

In [ ]:
pred_df = pd.DataFrame({
    'engine_id': test_df['engine_id'],
    'cycle': test_df['cycle'],
    'y_true': y_test,
    'y_pred': pred,
    })

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.scatterplot(data=pred_df.sample(min(4000, len(pred_df)), random_state=42), x='y_true', y='y_pred', alpha=0.35, s=12, ax=axes[0])
axes[0].plot([0, 130], [0, 130], linestyle='--', color='black', linewidth=1)
axes[0].set_title('Predicted vs True RUL')
axes[0].set_xlabel('True RUL')
axes[0].set_ylabel('Predicted RUL')

residual = pred_df['y_pred'] - pred_df['y_true']
sns.histplot(residual, bins=40, kde=True, ax=axes[1], color='#D95F02')
axes[1].set_title('Residual Distribution')
axes[1].set_xlabel('Prediction Error (y_pred - y_true)')

plt.tight_layout()
plot_path = REPORTS_DIR / '06_rf_baseline_diagnostics.png'
plt.savefig(plot_path, bbox_inches='tight')
plt.show()

with open(MODELS_DIR / 'rf_baseline.pkl', 'wb') as f:
    pickle.dump(rf, f)
pred_df.to_csv(MODELS_DIR / 'rf_test_predictions.csv', index=False)
with open(MODELS_DIR / 'rf_metrics.json', 'w', encoding='utf-8') as f:
    json.dump(metrics, f, indent=2)
(MODELS_DIR / 'rf_feature_columns.txt').write_text('\n'.join(feature_cols), encoding='utf-8')

print(f'Saved: {MODELS_DIR / "rf_baseline.pkl"}')
print(f'Saved: {MODELS_DIR / "rf_test_predictions.csv"}')
print(f'Saved: {MODELS_DIR / "rf_metrics.json"}')
print(f'Saved: {MODELS_DIR / "rf_feature_columns.txt"}')
print(f'Saved: {plot_path}')